<a href="https://colab.research.google.com/github/k9Sx3CC/01_first_look_and_discovery.ipynb/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

This project investigates whether machine learning can improve the prioritization of content refresh decisions compared with a simple rule-based baseline.

The goal is not to predict future traffic with certainty, but to identify pages that are more likely to be declining so that content editors can review them first. The model is intended to support editorial decision-making by producing a ranked refresh queue rather than automating content updates.

In [3]:
import json
import pandas as pd
import os
import subprocess
import sys
# Set the repository path
REPO_DIR = "/content/flyrank-ml-internship-starter" # Clone only if it doesn't already exist
if not os.path.exists(REPO_DIR): subprocess.run([ "git", "clone", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR ], check=True) # Always go to the absolute repository path
os.chdir(REPO_DIR) # Add current repo to Python's search path
sys.path.append(os.getcwd())
print("Current directory:", os.getcwd())
print("Contents:", os.listdir())
print(os.listdir("scripts"))
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py
!python scripts/04_evaluate_and_export.py

with open("outputs/model_results.json") as f:
    results = json.load(f)

print("Research Question")
print("-" * 40)
print("Target:", results["target"])
print("Best Model:", results["best_model"]["name"])
print("Validation:", results["split_strategy"])

Current directory: /content/flyrank-ml-internship-starter
Contents: ['SETUP.md', 'LICENSE', 'README.md', 'DATA_USE.md', 'work', '.git', 'submission', 'requirements.txt', '.gitignore', 'scripts', 'docs', 'GUIDE.md', 'data', 'skills', 'notebooks', 'CLAUDE.md', 'outputs', 'AGENTS.md', '.github']
['ml_utils.py', '04_evaluate_and_export.py', '05_build_pdf_report.py', 'run_all.py', '03_train_model.py', '02_baseline_score.py', '01_prepare_features.py']
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-s

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the anonymized content refresh dataset provided in the FlyRank ML Internship repository. The dataset contains approximately 30,000 pages with historical search, engagement, freshness, and content characteristics.

The feature preparation process created a structured feature vector containing numeric and categorical variables, including:

*  Search impressions and clicks
*  Click-through rate (CTR)
*  Average search position
*  Content age
*  Days since last update
*  Engagement metrics
*  Content characteristics

Client names, URLs, and other identifying information were excluded to preserve privacy. Only historical features that would be available before making a prediction were included in the final model.

In [4]:
import pandas as pd

features = pd.read_csv("data/processed/refresh_feature_vector.csv")

print("Dataset Summary")
print("-" * 40)
print("Rows:", len(features))
print("Columns:", len(features.columns))

features.head()

Dataset Summary
----------------------------------------
Rows: 30000
Columns: 52


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

he prediction target was is_declining_label, derived from the historical trend_direction variable. The target label was used only for training and evaluation and was not included as an input feature.

Three machine learning models were evaluated:

*  Logistic Regression
*  Decision Tree
*  Random Forest

Performance was compared with the Week 4 rule-based baseline.

Model evaluation used a client_holdout split so that pages from the same client did not appear in both training and testing sets. This provides a more realistic estimate of generalization to unseen clients.

A leakage audit confirmed that variables directly derived from the target, such as trend_direction and trend_pct, were excluded from the final feature set.

In [5]:
with open("outputs/model_results.json") as f:
    results = json.load(f)

summary = pd.DataFrame({
    "Item":[
        "Target",
        "Validation Split",
        "Training Rows",
        "Testing Rows",
        "Number of Features"
    ],
    "Value":[
        results["target"],
        results["split_strategy"],
        results["train_rows"],
        results["test_rows"],
        results["feature_count"]
    ]
})

summary

,Item,Value
0,Target,is_declining_label
1,Validation Split,client_holdout
2,Training Rows,27675
3,Testing Rows,2325
4,Number of Features,52


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The Random Forest produced the strongest measured performance under the client-holdout evaluation.

| Method              | Precision@50 |
| ------------------- | -----------: |
| Baseline            |     **0.24** |
| Logistic Regression |     **0.40** |
| Decision Tree       |     **0.58** |
| Random Forest       |     **0.74** |

The Random Forest improved Precision@50 from 0.24 to 0.74 under the same evaluation strategy. These results suggest that the model can better prioritize pages for manual review within this dataset. Additional evaluation on future data would strengthen confidence in deployment performance.

In [6]:
import json
import pandas as pd

with open("outputs/model_results.json") as f:
    results = json.load(f)

comparison = pd.DataFrame({
    "Method":[
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Precision@50":[
        results["baseline"]["baseline_precision_at_50"],
        results["models"]["logistic_regression"]["precision_at_50"],
        results["models"]["decision_tree"]["precision_at_50"],
        results["models"]["random_forest"]["precision_at_50"]
    ],
    "ROC AUC":[
        results["baseline"]["baseline_roc_auc"],
        results["models"]["logistic_regression"]["roc_auc"],
        results["models"]["decision_tree"]["roc_auc"],
        results["models"]["random_forest"]["roc_auc"]
    ]
})

comparison

,Method,Precision@50,ROC AUC
0,Baseline,0.24,0.626892
1,Logistic Regression,0.40,0.700291
2,Decision Tree,0.58,0.741520
3,Random Forest,0.74,0.750030


## 5. Limitations

*What this work cannot claim.*

This work has several limitations.

*  The model was evaluated using historical data from one anonymized dataset.
*  Performance may change if search engine algorithms or user behavior change.
*  The model identifies pages for review but does not determine why a page is declining.
*  The recommendations should not be interpreted as proof that updating a page will improve performance.
*  Editorial expertise remains necessary before making content changes.

The model should therefore be used as a decision-support tool rather than an automated publishing system.

In [7]:
import json

with open("outputs/model_results.json") as f:
    results = json.load(f)

print("Evaluation Summary")
print("-"*40)
print("Validation Strategy :", results["split_strategy"])
print("Input Rows          :", results["input_rows"])
print("Positive Rate       :", round(results["target_positive_rate"],3))
print("Best Model          :", results["best_model"]["name"])

Evaluation Summary
----------------------------------------
Validation Strategy : client_holdout
Input Rows          : 30000
Positive Rate       : 0.542
Best Model          : random_forest


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The final output is a ranked refresh queue ordered by predicted refresh priority.

Editors should review the highest-ranked pages first because they have the highest measured refresh scores. Each recommendation includes reason codes that explain why the page was selected.

Typical recommendation reasons include:

* declining_with_demand – the page is declining while still receiving search demand.
* low_ctr_visible_page – the page ranks well but has a relatively low click-through rate.
* model_decline_risk – the model predicts an increased likelihood of decline.
* visible_page – the page already receives search visibility and may benefit from improvements.
* refresh_and_review_ctr – both content quality and click-through performance should be reviewed.

These recommendations are intended to prioritize editorial review and should always be confirmed by a human before any content changes are made.

In [8]:
import pandas as pd

queue = pd.read_csv("outputs/refresh_queue.csv")

cols = [
    "final_rank",
    "final_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes"
]

queue[cols].head(10)
queue["suggested_action"].value_counts()

,count
suggested_action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

he following outputs were generated during the project and can be included in the final paper:

* refresh_queue.csv – ranked list of recommended pages
* refresh_queue_sample.csv – sample recommendations
* model_results.json – evaluation metrics
model_report.md – model summary
* summary.json – project summary
* Feature Importance chart – most influential model features
* Model comparison table – Baseline, Logistic Regression, Decision Tree, and Random Forest
* Refresh queue preview – example of the highest-ranked recommendations

Together, these artifacts document the methodology, model performance, feature importance, and final decision-support recommendations used throughout the project.

In [9]:
import os

print("Generated Artifacts")
print("-"*40)

for file in sorted(os.listdir("outputs")):
    print(file)

Generated Artifacts
----------------------------------------
charts
model_report.md
model_results.json
refresh_queue.csv
refresh_queue_sample.csv
summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
